# JailbreakBench Survey — Phase 2 / 3 runner (Colab Pro)

This notebook drives the Phase 2 (defended ASR) and Phase 3 (benign refusal
rate) experiments on a Colab Pro GPU. The target models are loaded
**locally** with `vLLM` because Together AI no longer serves Vicuna-13B-v1.5
or Llama-2-7B-chat-hf via its serverless API.

**Recommended GPU**: T4 or L4. Avoid A100 unless you actually need it — it
burns through Pro compute units 6× faster.

**Compute budget**: ~5-10 GPU-hours total for all experiments.

Before running, set a Colab secret named `TOGETHER_API_KEY` (sidebar → key
icon → Add new secret). The judge calls go to Together AI; the target-model
queries do not.

## 1. Verify GPU availability

In [ ]:
!nvidia-smi

## 2. Clone the project repository

The repo is currently private. The clone cell below uses an HTTPS URL and
will prompt for a GitHub username + Personal Access Token on first run, or
read them from `~/.netrc` if present. Once we make the repo public for the
final submission this prompt goes away.

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/imzjes/jbb-survey.git'
REPO_DIR = '/content/jbb-survey'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!ls

## 3. Install dependencies (cloud + vLLM extras)

In [ ]:
# JBB requires Python 3.10/3.11; Colab's default is 3.12. Stand up a 3.11 venv with uv.
!pip install --quiet uv
!uv venv --python 3.11 --clear /content/jbb-venv
!uv pip install --python /content/jbb-venv/bin/python --quiet -r requirements.txt -r requirements-vllm.txt
!/content/jbb-venv/bin/python -c 'import jailbreakbench; from jailbreakbench.llm.vllm import LLMvLLM; print("vLLM backend OK")'

## 3a. Patch JBB's vllm wrapper for newer vllm

JBB 1.0.0 imports `destroy_model_parallel` from a path that vllm 0.5+
relocated. Apply a small in-place patch so the wrapper imports cleanly
under our pinned vllm 0.6.6.

In [ ]:
import pathlib

site = pathlib.Path('/content/jbb-venv/lib/python3.11/site-packages')

# Patch 1: vllm.model_executor.parallel_utils moved in vllm 0.5+.
vllm_wrap = site / 'jailbreakbench/llm/vllm.py'
src = vllm_wrap.read_text()
old_import = "from vllm.model_executor.parallel_utils.parallel_state import destroy_model_parallel"
new_import = (
    "try:\n"
    "    from vllm.model_executor.parallel_utils.parallel_state import destroy_model_parallel\n"
    "except ImportError:\n"
    "    try:\n"
    "        from vllm.distributed.parallel_state import destroy_model_parallel\n"
    "    except ImportError:\n"
    "        def destroy_model_parallel():\n"
    "            pass"
)

# Patch 2: cap max_model_len so the wrapper fits on smaller GPUs.
old_call = "self.model = vllm.LLM(model=self.hf_model_name)"
new_call = (
    "self.model = vllm.LLM(\n"
    "            model=self.hf_model_name,\n"
    "            max_model_len=2048,\n"
    "            gpu_memory_utilization=0.90,\n"
    "        )"
)

changed = False
if old_import in src:
    src = src.replace(old_import, new_import); changed = True
if old_call in src:
    src = src.replace(old_call, new_call); changed = True
if changed:
    vllm_wrap.write_text(src)
    print('patched JBB vllm wrapper')

# Patch 3: hard-coded local paths in defense hparams -> public HF model id.
hparams = site / 'jailbreakbench/defenses/defenselib/defense_hparams.py'
src = hparams.read_text()
old_path = '/shared_data0/arobey1/llama-2-7b-chat-hf'
new_path = 'meta-llama/Llama-2-7b-chat-hf'
if old_path in src:
    src = src.replace(old_path, new_path)
    hparams.write_text(src)
    print('patched defense_hparams paths')

# Patch 4: substitute de-listed Together model strings + add retries +
# tolerate occasional rate-limit failures (default to safe/non-jailbroken).
classifier = site / 'jailbreakbench/classifier.py'
src = classifier.read_text()
substitute = '"together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo"'
classifier_changed = False
for old_model in (
    '"together_ai/Meta-Llama/Llama-Guard-7b"',
    '"together_ai/meta-llama/Llama-3-70b-chat-hf"',
    '"together_ai/meta-llama/Llama-3-8b-chat-hf"',
):
    if old_model in src:
        src = src.replace(old_model, substitute)
        classifier_changed = True

# LlamaGuard1JailbreakJudge.classify_responses: add retries + try/except.
old_lg = '''outputs = litellm.batch_completion(
            model=self.LITELLM_MODEL, messages=classifier_inputs, api_key=self._api_key, temperature=0
        )

        classifications = []
        for output in outputs:
            content = output["choices"][0]["message"].content.strip()
            classifications.append(content[:4] != "safe")'''
new_lg = '''outputs = litellm.batch_completion(
            model=self.LITELLM_MODEL, messages=classifier_inputs, api_key=self._api_key, temperature=0,
            num_retries=5,
        )

        classifications = []
        for output in outputs:
            try:
                content = output["choices"][0]["message"].content.strip()
                classifications.append(content[:4] != "safe")
            except (TypeError, KeyError, AttributeError):
                classifications.append(False)'''
if old_lg in src:
    src = src.replace(old_lg, new_lg)
    classifier_changed = True

# Llama3Classifier.classify_responses: same treatment.
old_l3 = '''outputs = litellm.batch_completion(
            model=self.LITELLM_MODEL, messages=classifier_inputs, api_key=self._api_key, temperature=0
        )
        classifications = []
        for output in outputs:
            content = output["choices"][0]["message"].content.strip()
            classifications.append(self.POSITIVE_ANSWER in content)'''
new_l3 = '''outputs = litellm.batch_completion(
            model=self.LITELLM_MODEL, messages=classifier_inputs, api_key=self._api_key, temperature=0,
            num_retries=5,
        )
        classifications = []
        for output in outputs:
            try:
                content = output["choices"][0]["message"].content.strip()
                classifications.append(self.POSITIVE_ANSWER in content)
            except (TypeError, KeyError, AttributeError):
                classifications.append(False)'''
if old_l3 in src:
    src = src.replace(old_l3, new_l3)
    classifier_changed = True

if classifier_changed:
    classifier.write_text(src)
    print('patched JBB classifier (model + retries + error tolerance)')

!/content/jbb-venv/bin/python -c 'import jailbreakbench, vllm; from jailbreakbench.llm.vllm import LLMvLLM; print("vLLM backend OK, vllm:", vllm.__version__)'

## 4. Load API tokens from Colab secrets

`TOGETHER_API_KEY` is needed for the judge calls. `HF_TOKEN` is needed
because Llama-2-7B is a gated HuggingFace model — request access at
https://huggingface.co/meta-llama/Llama-2-7b-chat-hf, then create a Read
token in HF Settings → Access Tokens, and add it as a Colab secret.

In [ ]:
import os
from google.colab import userdata

os.environ['TOGETHER_API_KEY'] = userdata.get('TOGETHER_API_KEY')
assert os.environ['TOGETHER_API_KEY'], 'Set the TOGETHER_API_KEY Colab secret'

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']  # vllm reads this name
assert os.environ['HF_TOKEN'], 'Set the HF_TOKEN Colab secret'

print('TOGETHER_API_KEY length =', len(os.environ['TOGETHER_API_KEY']))
print('HF_TOKEN length =', len(os.environ['HF_TOKEN']))

# All subsequent script invocations should use the 3.11 venv:
PY = '/content/jbb-venv/bin/python'

## 5. Phase 1 — rejudge artifacts (no GPU needed)

Cheap and fast: ~200 judge calls. Run this first as an end-to-end sanity
check before paying for the GPU phases.

In [ ]:
!/content/jbb-venv/bin/python phase1_baseline_asr.py --limit 5     # smoke test

In [ ]:
!/content/jbb-venv/bin/python phase1_baseline_asr.py     # full run

## 6. Phase 2 — defended ASR (GPU)

Smoke test first (5 prompts × 1 model × 1 attack × 1 defense). If this
completes cleanly, run the full sweep below.

In [ ]:
!/content/jbb-venv/bin/python phase2_defense_asr.py --backend vllm \
    --models llama-2-7b-chat-hf --attacks PAIR \
    --defenses SmoothLLM --limit 5

In [ ]:
# Full Phase 2 sweep. All defenses run locally via vLLM on this Colab.
!/content/jbb-venv/bin/python phase2_defense_asr.py --backend vllm \
    --defenses SmoothLLM PerplexityFilter EraseAndCheck

## 7. Phase 3 — benign refusal rate (GPU)

In [ ]:
!/content/jbb-venv/bin/python phase3_benign_refusal.py --backend vllm \
    --models llama-2-7b-chat-hf --defenses SmoothLLM --limit 5

In [ ]:
!/content/jbb-venv/bin/python phase3_benign_refusal.py --backend vllm \
    --defenses SmoothLLM PerplexityFilter EraseAndCheck \
    --include-undefended

## 8. Generate figures + LaTeX tables for the report

In [ ]:
!/content/jbb-venv/bin/python make_figures.py
!ls results/

## 9. Pull results back to your laptop

Easiest: `git add results/` (the .gitignore excludes them by default — drop
the relevant files into a separate `report-assets/` dir before committing
if you want them in the repo), then `git push`. Or right-click the
`results/` folder in the Colab file browser and download as zip.